# 4. Investigate event-level errors, then decide whether to open final assessment
No point adjustment. One incident can match at most one fault. Duplicate incidents
remain workload. Detection delay uses the observable-onset proxy, with physical
onset retained in truth. Warning opportunity requires at least three consecutive
observed Rx readings after observable onset and at least 30 minutes before impact.
This denominator does not depend on detector scores or the chosen debounce policy.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
# Resolve relative configured output paths consistently from any notebook.
import os

os.chdir(ROOT)
from optical_anomaly.pipeline import prepare, develop, final_evaluation

CONFIG_PATH = ROOT / "configs/config.yaml"
RUN = prepare(CONFIG_PATH)
settings = json.loads((RUN / "settings.json").read_text())
start = pd.Timestamp("2025-01-01", tz="UTC")
boundaries = [
    start + pd.Timedelta(days=settings["generator"]["days"] * f)
    for f in settings["splits"]
]

In [ ]:
outcomes = pd.read_csv(RUN / "validation_faults.csv")
display(
    outcomes.groupby("fault_type").agg(
        faults=("detected", "size"),
        detected=("detected", "sum"),
        opportunities=("opportunity", "sum"),
        pre_impact=("pre_impact", "sum"),
    )
)
display(outcomes.loc[~outcomes.detected])
display(pd.read_csv(RUN / "validation_incidents.csv").head(12))

In [ ]:
OPEN_FINAL_TEST = False
if OPEN_FINAL_TEST:
    display(pd.Series(final_evaluation(RUN)))
else:
    print("Final performance assessment remains unopened.")

Final assessment reuses the frozen model and policy. Code/data/model checksums
prevent accidental changes; a one-opening marker prevents accidental repeat use.
These are local safeguards, not a security boundary. Once inspected, final data
cannot serve as an untouched test for further tuning. Faults crossing split
boundaries are reported separately, not silently treated as misses or successes.